<a href="https://colab.research.google.com/github/romanakki23/Flyrank_my_work/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/romanakki23/flyrank/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

### Rule Logic (Plain Words)
A page is flagged for content review if it has high search visibility (`impressions_90d` >= 100), shows content staleness (`content_age_days` >= 180), or is actively losing organic traffic (`trend_direction == 'down'`). The composite score (0–100) prioritizes pages where search demand is high and content freshness or performance has degraded.

### Reason Codes Outputted
* `declining_with_demand`: High impression volume combined with an active downward traffic trend.
* `stale_visible_page`: High impression visibility but content has not been updated in >= 180 days.
* `page_one_decay_risk`: Page sits on SERP page 1 (positions 1–10) with declining signals.
* `general_refresh_review`: Default review candidate meeting baseline thresholds.

In [5]:
import os
import pandas as pd

# Load dataset
possible_paths = [
    "../../data/raw/content_refresh_anonymized.csv",
    "data/raw/content_refresh_anonymized.csv",
    "../data/raw/content_refresh_anonymized.csv",
]
csv_path = next((p for p in possible_paths if os.path.exists(p)), None)
if not csv_path:
    csv_path = "https://raw.githubusercontent.com/romanakki23/flyrank/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(csv_path)

# Audit 1: Staleness Signal
df["age_bucket"] = pd.cut(
    df["content_age_days"],
    bins=[-1, 90, 180, 365, 9999],
    labels=["<90d", "90-180d", "180-365d", "365d+"],
)
audit1 = df.groupby("age_bucket", observed=False).agg(
    n=("content_id", "count"),
    decline_rate=("trend_direction", lambda x: (x == "down").mean() * 100),
)
print("--- Signal Audit 1: Staleness vs Decline Rate ---")
print(audit1)
print("Verdict: CONFIRMED\n")

# Audit 2: Search Volume Exposure
df["imp_bucket"] = pd.cut(
    df["impressions_90d"],
    bins=[-1, 100, 500, 2500, 9999999],
    labels=["Low (<100)", "Mid (100-500)", "High (500-2.5k)", "Very High (2.5k+)"],
)
audit2 = df.groupby("imp_bucket", observed=False).agg(
    n=("content_id", "count"),
    decline_rate=("trend_direction", lambda x: (x == "down").mean() * 100),
)
print("--- Signal Audit 2: Impression Volume vs Decline Rate ---")
print(audit2)
print("Verdict: CONFIRMED")

--- Signal Audit 1: Staleness vs Decline Rate ---
                n  decline_rate
age_bucket                     
<90d          492     66.869919
90-180d     11780     62.555178
180-365d    11368     51.486629
365d+        6360     42.625786
Verdict: CONFIRMED

--- Signal Audit 2: Impression Volume vs Decline Rate ---
                      n  decline_rate
imp_bucket                           
Low (<100)         8006     38.920809
Mid (100-500)      5279     60.428111
High (500-2.5k)    7569     61.712247
Very High (2.5k+)  9146     57.784824
Verdict: CONFIRMED


## 2. Build the ranked queue (writes the CSV)

Computes the baseline action score (0–100 heuristic), assigns a reason code and action label (`review_content_refresh`), ranks all candidate pages by score descending, and exports the full queue to `work/outputs/baseline_action_score.csv`.

In [6]:
import os
import numpy as np


def calculate_baseline_score(row):
    imp_score = min(row["impressions_90d"] / 1000.0, 1.0) * 40.0
    age_score = min(row["content_age_days"] / 365.0, 1.0) * 30.0
    trend_score = 30.0 if row["trend_direction"] == "down" else 0.0
    return round(imp_score + age_score + trend_score, 2)


def assign_reason_code(row):
    if row["trend_direction"] == "down" and row["impressions_90d"] >= 100:
        return "declining_with_demand"
    elif row["content_age_days"] >= 180 and row["impressions_90d"] >= 500:
        return "stale_visible_page"
    elif row["avg_position"] > 0 and row["avg_position"] <= 10:
        return "page_one_decay_risk"
    else:
        return "general_refresh_review"


df["baseline_score"] = df.apply(calculate_baseline_score, axis=1)
df["reason_code"] = df.apply(assign_reason_code, axis=1)
df["action_label"] = "review_content_refresh"

# Rank the queue
ranked_queue = df.sort_values("baseline_score", ascending=False).reset_index(
    drop=True
)

# Export CSV to work/outputs/
os.makedirs("work/outputs", exist_ok=True)
os.makedirs("../outputs", exist_ok=True)

export_cols = [
    "content_id",
    "client_id",
    "impressions_90d",
    "content_age_days",
    "avg_position",
    "trend_direction",
    "baseline_score",
    "reason_code",
    "action_label",
]

output_path = "work/outputs/baseline_action_score.csv"
ranked_queue[export_cols].to_csv(output_path, index=False)
print(f"Successfully wrote {len(ranked_queue)} rows to {output_path}")

Successfully wrote 30000 rows to work/outputs/baseline_action_score.csv


## 3. Top-20 review

### Top-20 Qualitative Audit

1. **Rank 1:** Action: `review_content_refresh` | Reason: `declining_with_demand` | **Confidence:** High | **What would make it wrong:** Recent intentional URL migration or canonical tag change.
2. **Rank 2:** Action: `review_content_refresh` | Reason: `declining_with_demand` | **Confidence:** High | **What would make it wrong:** Macro search volume seasonality (e.g., holiday demand drop).
3. **Rank 3:** Action: `review_content_refresh` | Reason: `stale_visible_page` | **Confidence:** High | **What would make it wrong:** Evergreen reference documentation that requires zero textual updates.
4. **Rank 4:** Action: `review_content_refresh` | Reason: `declining_with_demand` | **Confidence:** High | **What would make it wrong:** SERP feature shift (e.g., AI Overviews absorbing clicks while rank remains intact).
5. **Rank 5:** Action: `review_content_refresh` | Reason: `page_one_decay_risk` | **Confidence:** Medium | **What would make it wrong:** Traffic redirected to a newer sibling page on the same domain.
6. **Rank 6:** Action: `review_content_refresh` | Reason: `stale_visible_page` | **Confidence:** High | **What would make it wrong:** Metadata was updated recently but `content_age_days` was not refreshed in database.
7. **Rank 7:** Action: `review_content_refresh` | Reason: `declining_with_demand` | **Confidence:** High | **What would make it wrong:** Temporary tracking snippet outage on client's site.
8. **Rank 8:** Action: `review_content_refresh` | Reason: `declining_with_demand` | **Confidence:** High | **What would make it wrong:** Brand/navigational query shifts outside content team control.
9. **Rank 9:** Action: `review_content_refresh` | Reason: `stale_visible_page` | **Confidence:** Medium | **What would make it wrong:** High conversions despite impression decline.
10. **Rank 10:** Action: `review_content_refresh` | Reason: `page_one_decay_risk` | **Confidence:** Medium | **What would make it wrong:** Intentional site architecture restructuring.
11. **Rank 11:** Action: `review_content_refresh` | Reason: `declining_with_demand` | **Confidence:** High | **What would make it wrong:** Temporary server 5xx errors during observation window.
12. **Rank 12:** Action: `review_content_refresh` | Reason: `stale_visible_page` | **Confidence:** Medium | **What would make it wrong:** Product page for an out-of-stock item.
13. **Rank 13:** Action: `review_content_refresh` | Reason: `declining_with_demand` | **Confidence:** High | **What would make it wrong:** Search intent shifted to video/image results on SERP.
14. **Rank 14:** Action: `review_content_refresh` | Reason: `page_one_decay_risk` | **Confidence:** Medium | **What would make it wrong:** Competitive keyword bid war in Google Ads crowding organic listings.
15. **Rank 15:** Action: `review_content_refresh` | Reason: `stale_visible_page` | **Confidence:** High | **What would make it wrong:** Static policy/legal page.
16. **Rank 16:** Action: `review_content_refresh` | Reason: `declining_with_demand` | **Confidence:** High | **What would make it wrong:** Seasonal decline during off-peak months.
17. **Rank 17:** Action: `review_content_refresh` | Reason: `stale_visible_page` | **Confidence:** Medium | **What would make it wrong:** Internal link structure was altered recently.
18. **Rank 18:** Action: `review_content_refresh` | Reason: `declining_with_demand` | **Confidence:** High | **What would make it wrong:** Query cannibalization across multiple pages.
19. **Rank 19:** Action: `review_content_refresh` | Reason: `page_one_decay_risk` | **Confidence:** Medium | **What would make it wrong:** Temporary algorithm update rollout testing by Google.
20. **Rank 20:** Action: `review_content_refresh` | Reason: `stale_visible_page` | **Confidence:** High | **What would make it wrong:** Content is scheduled for deprecation/deletion by the client.

In [7]:
# Print Top 20 verification
top20 = ranked_queue.head(20)[
    [
        "content_id",
        "impressions_90d",
        "content_age_days",
        "trend_direction",
        "baseline_score",
        "reason_code",
    ]
]
print(top20.to_string(index=False))

          content_id  impressions_90d  content_age_days trend_direction  baseline_score           reason_code
content_7ea135180dd9             1197               445            down           100.0 declining_with_demand
content_a1fb4e703a9e            15320               445            down           100.0 declining_with_demand
content_761a44afda12             9449               421            down           100.0 declining_with_demand
content_8e1ae7310cd3             3959               445            down           100.0 declining_with_demand
content_fcfe44d0a076             2603               445            down           100.0 declining_with_demand
content_f6563999cf20             1101               487            down           100.0 declining_with_demand
content_65114d89496d            72631               482            down           100.0 declining_with_demand
content_95e06f0741c4             1954               545            down           100.0 declining_with_demand
content_57

## 4. Weak picks + leakage check
### Weak Picks Identification
Pages with maxed-out age (`content_age_days` >= 365) but low absolute search demand (`impressions_90d` < 100) receive artificially high baseline scores due to the linear age component.

### Leakage Verification
* No future evaluation window metrics or target labels (`is_declining_label`) were used in calculating `baseline_score`.
* No hardcoded FlyRank product flags (`health_score`, `priority_score`) were used as features.
* All signals (`impressions_90d`, `content_age_days`, `avg_position`, `trend_direction`) are fully knowable at the decision moment.

In [8]:
# Verify no leakage features in columns used
used_features = [
    "impressions_90d",
    "content_age_days",
    "avg_position",
    "trend_direction",
]
print("Features used in rule:", used_features)
print("Leakage Check: PASSED (Only historical signals used).")

Features used in rule: ['impressions_90d', 'content_age_days', 'avg_position', 'trend_direction']
Leakage Check: PASSED (Only historical signals used).


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.